# Import

In [17]:
import os
import shutil
import re
import pickle


def ensure_dirs(*dirs):
    for d in dirs:
        os.makedirs(d, exist_ok=True)

def parse_run_log(log_path):
    success, fail = [], []
    with open(log_path, "r") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        match = re.match(r"==== Processing (.+)\.mod ====", line)
        if match:
            hash_id = match.group(1)
            next_line = lines[i + 1] if i + 1 < len(lines) else ""
            if "Successfully processed" in next_line:
                success.append(hash_id)
            elif "failed" in next_line.lower():
                fail.append(hash_id)
    return success, fail

def move_logs(hashes, src_dir, dst_dir):
    for h in hashes:
        src = os.path.join(src_dir, f"{h}.log")
        dst = os.path.join(dst_dir, f"{h}.log")
        if os.path.exists(src):
            shutil.copy2(src, dst)

def copy_failed_mods(failed_log_dir, mod_src_dir, mod_dst_dir):
    failed_hashes = [f.replace(".log", "") for f in os.listdir(failed_log_dir) if f.endswith(".log")]
    copied = 0
    for h in failed_hashes:
        src = os.path.join(mod_src_dir, f"{h}.mod")
        dst = os.path.join(mod_dst_dir, f"{h}.mod")
        if os.path.exists(src):
            shutil.copy2(src, dst)
            copied += 1
        else:
            print(f"⚠️ Missing .mod for {h}")
    return copied

def move_failed_files(nmodl_src, log_src, dst_root):
    nmodl_dst = os.path.join(dst_root, "nmodl")
    log_dst = os.path.join(dst_root, "log")
    ensure_dirs(nmodl_dst, log_dst)

    for f in os.listdir(nmodl_src):
        shutil.move(os.path.join(nmodl_src, f), os.path.join(nmodl_dst, f))
    for f in os.listdir(log_src):
        shutil.move(os.path.join(log_src, f), os.path.join(log_dst, f))

    os.rmdir(nmodl_src)
    os.rmdir(log_src)

    print(f"✅ Moved nmodl-failed → {nmodl_dst}")
    print(f"✅ Moved logs/failed → {log_dst}")


def load_excluded_hashes(pipeline_dir):
    files = ["failed_excluded.pkl", "failed_neither.pkl"]
    excluded = set()
    for fname in files:
        with open(os.path.join(pipeline_dir, fname), "rb") as f:
            excluded.update(pickle.load(f))
    return excluded


# === Main execution ===
log_folder = "/gpfs/gibbs/project/mcdougal/imc33/mod-extract/logs"
run_log_path = os.path.join(log_folder, "run_log.txt")
success_dir = os.path.join(log_folder, "successful")
fail_dir = os.path.join(log_folder, "failed")
ensure_dirs(success_dir, fail_dir)

# Load and parse hashes
success_hashes, fail_hashes = parse_run_log(run_log_path)

# Exclude specified failed hashes
pipeline_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "pipeline"))
excluded_hashes = load_excluded_hashes(pipeline_dir)
fail_hashes = [h for h in fail_hashes if h not in excluded_hashes]

# Move logs
move_logs(success_hashes, log_folder, success_dir)
move_logs(fail_hashes, log_folder, fail_dir)

print(f"✅ Moved {len(success_hashes)} successful logs")
print(f"❌ Moved {len(fail_hashes)} filtered failed logs")

# Copy .mod files from filtered failed logs
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
failed_log_dir = os.path.join(project_root, "logs", "failed")
mod_src_dir = os.path.join(project_root, "data", "raw", "nmodl")
mod_dst_dir = os.path.join(project_root, "data", "raw", "nmodl-failed")
ensure_dirs(mod_dst_dir)

copied_count = copy_failed_mods(failed_log_dir, mod_src_dir, mod_dst_dir)
print(f"✅ Copied {copied_count} .mod files to '{mod_dst_dir}'")


✅ Moved 426 successful logs
❌ Moved 128 filtered failed logs
✅ Copied 128 .mod files to '/gpfs/gibbs/project/mcdougal/imc33/mod-extract/data/raw/nmodl-failed'


In [18]:
!git add .
!git commit -m "excluded neither and subtypes we did not include in the analysis from the failed file list"
!git push

[main 6ae6b31] excluded neither and subtypes we did not include in the analysis from the failed file list
 2 files changed, 111 insertions(+), 578 deletions(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 64 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.05 KiB | 116.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To github.com:innacohen/mod-extract.git
   e22cd97..6ae6b31  main -> main
